<a href="https://colab.research.google.com/github/euleralencar/projeto_ciencia_dados_ibmec/blob/main/02_2_numerical_pipeline_hands_on.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalhando com dados numéricos

No notebook anterior, treinamos um modelo de k-nearest neighbors com alguns
dados.

No entanto, simplificamos demais o procedimento ao carregar um conjunto de
dados que continha exclusivamente dados numéricos. Além disso, usamos conjuntos
de dados que já estavam divididos em conjuntos de treino e teste.

Neste notebook, nosso objetivo é:

* identificar dados numéricos em um conjunto de dados heterogêneo;
* selecionar o subconjunto de colunas correspondente aos dados numéricos;
* usar um utilitário do scikit-learn para separar os dados em conjuntos de
  treino e teste;
* treinar e avaliar um modelo mais complexo do scikit-learn.

Começamos carregando o conjunto de dados do censo norte-americano usado durante
a exploração dos dados.

## Carregando o conjunto de dados completo

Assim como no notebook anterior, usamos o pandas para abrir o arquivo CSV em um
dataframe do pandas.

In [1]:
import pandas as pd

adult_census = pd.read_csv(
    "https://raw.githubusercontent.com/INRIA/scikit-learn-mooc/main/datasets/adult-census.csv"
)

# remove a coluna duplicada `"education-num"` conforme indicado no primeiro notebook
adult_census = adult_census.drop(columns="education-num")
adult_census

,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,11th,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,HS-grad,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,Assoc-acdm,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,Some-college,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,Some-college,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
48838,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
48839,58,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
48840,22,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


O próximo passo separa o alvo dos dados. Realizamos o mesmo procedimento no
notebook anterior.

In [2]:
data, target = adult_census.drop(columns="class"), adult_census["class"]

In [3]:
data

,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,11th,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,HS-grad,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,Assoc-acdm,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,Some-college,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,?,Some-college,Never-married,?,Own-child,White,Female,0,0,30,United-States
...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States
48838,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States
48839,58,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States
48840,22,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States


In [4]:
target

,class
0,<=50K
1,<=50K
2,>50K
3,>50K
4,<=50K
...,...
48837,<=50K
48838,>50K
48839,<=50K
48840,<=50K


<div class="admonition note alert alert-info">
<p class="first admonition-title" style="font-weight: bold;">Nota</p>
<p class="last">Aqui e mais adiante, usamos os nomes <tt class="docutils
literal">data</tt> e <tt class="docutils literal">target</tt> para sermos
explícitos. Na documentação do scikit-learn, <tt class="docutils
literal">data</tt> é comumente chamado de <tt class="docutils literal">X</tt> e
<tt class="docutils literal">target</tt> é comumente chamado de <tt
class="docutils literal">y</tt>.</p>
</div>

Neste ponto, podemos focar nos dados que queremos usar para treinar nosso
modelo preditivo.

## Identificar dados numéricos

Dados numéricos são representados por números. Eles estão ligados a dados
mensuráveis (quantitativos), como a idade ou o número de horas que uma pessoa
trabalha por semana.

Modelos preditivos são nativamente projetados para trabalhar com dados
numéricos. Além disso, dados numéricos geralmente exigem pouquíssimo trabalho
antes de começar o treinamento.

A primeira tarefa aqui é identificar dados numéricos em nosso conjunto de
dados.

<div class="admonition caution alert alert-warning">
<p class="first admonition-title" style="font-weight: bold;">Cuidado!</p>
<p class="last">Dados numéricos são representados por números, mas nem todo
número representa dados numéricos. Categorias podem já estar codificadas com
números e você pode precisar identificar essas features.</p>
</div>

Assim, podemos verificar o tipo de dado de cada coluna do conjunto de dados.

In [5]:
data.dtypes

,0
age,int64
workclass,object
education,object
marital-status,object
occupation,object
relationship,object
race,object
sex,object
capital-gain,int64
capital-loss,int64


Parece que temos apenas dois tipos de dados: `int64` e `object`. Podemos
confirmar isso verificando os tipos de dados únicos.

In [6]:
data.dtypes.unique()

array([dtype('int64'), dtype('O')], dtype=object)

De fato, os únicos dois tipos no conjunto de dados são o inteiro `int64` e
`object`. Podemos observar as primeiras linhas do dataframe para entender o
significado do tipo de dado `object`.

In [7]:
data

,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,11th,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,HS-grad,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,Assoc-acdm,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,Some-college,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,?,Some-college,Never-married,?,Own-child,White,Female,0,0,30,United-States
...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States
48838,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States
48839,58,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States
48840,22,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States


Vemos que o tipo de dado `object` corresponde a colunas que contêm strings.
Como vimos na seção de exploração, essas colunas contêm categorias, e veremos
mais adiante como lidar com elas. Podemos selecionar as colunas que contêm
inteiros e verificar seu conteúdo.

In [8]:
numerical_columns = ["age", "capital-gain", "capital-loss", "hours-per-week"]
data[numerical_columns]

,age,capital-gain,capital-loss,hours-per-week
0,25,0,0,40
1,38,0,0,50
2,28,0,0,40
3,44,7688,0,40
4,18,0,0,30
...,...,...,...,...
48837,27,0,0,38
48838,40,0,0,40
48839,58,0,0,40
48840,22,0,0,20


Agora que limitamos o conjunto de dados apenas às colunas numéricas, podemos
analisar esses números para descobrir o que representam. Podemos identificar
dois tipos de uso.

A primeira coluna, `"age"`, é autoexplicativa. Podemos notar que os valores são
contínuos, ou seja, podem assumir qualquer número dentro de um determinado
intervalo. Vamos descobrir qual é esse intervalo:

In [9]:
data["age"].describe()

,age
count,48842.000000
mean,38.643585
std,13.710510
min,17.000000
25%,28.000000
50%,37.000000
75%,48.000000
max,90.000000


Podemos ver que a idade varia entre 17 e 90 anos.

Poderíamos estender nossa análise e descobriríamos que `"capital-gain"`,
`"capital-loss"` e `"hours-per-week"` também representam dados quantitativos.

Agora, armazenamos o subconjunto de colunas numéricas em um novo dataframe.

In [10]:
data_numeric = data[numerical_columns]

## Divisão treino-teste do conjunto de dados

No notebook anterior, carregamos dois conjuntos de dados separados: um de
treino e um de teste. No entanto, ter conjuntos de dados separados em dois
arquivos distintos é incomum: na maioria das vezes, temos um único arquivo
contendo todos os dados que precisamos dividir depois de carregados na memória.

O scikit-learn fornece a função utilitária
`sklearn.model_selection.train_test_split`, usada para dividir automaticamente
o conjunto de dados em dois subconjuntos.

In [11]:
from sklearn.model_selection import train_test_split

data_train, data_test, target_train, target_test = train_test_split(
    data_numeric, target, random_state=42, test_size=0.25
)

<div class="admonition tip alert alert-warning">
<p class="first admonition-title" style="font-weight: bold;">Dica</p>
<p class="last">No scikit-learn, definir o parâmetro <tt class="docutils
literal">random_state</tt> permite obter resultados determinísticos quando
usamos um gerador de números aleatórios. No caso do <tt class="docutils
literal">train_test_split</tt>, a aleatoriedade vem do embaralhamento dos
dados, que decide como o conjunto de dados é dividido em um conjunto de treino
e um de teste).</p>
</div>

Ao chamar a função `train_test_split`, especificamos que gostaríamos de ter 25%
das amostras no conjunto de teste, enquanto as amostras restantes (75%) são
atribuídas ao conjunto de treino. Podemos verificar rapidamente se obtivemos o
que esperávamos.

In [12]:
print(
    f"Número de amostras no teste: {data_test.shape[0]} => "
    f"{data_test.shape[0] / data_numeric.shape[0] * 100:.1f}% do "
    "conjunto original"
)

Número de amostras no teste: 12211 => 25.0% do conjunto original


In [13]:
print(
    f"Número de amostras no treino: {data_train.shape[0]} => "
    f"{data_train.shape[0] / data_numeric.shape[0] * 100:.1f}% do "
    "conjunto original"
)

Número de amostras no treino: 36631 => 75.0% do conjunto original


No notebook anterior, usamos um modelo de k-nearest neighbors. Embora esse
modelo seja intuitivo de entender, ele não é amplamente usado na prática.
Agora, usamos um modelo mais útil, chamado regressão logística, que pertence à
família dos modelos lineares.

<div class="admonition note alert alert-info">
<p class="first admonition-title" style="font-weight: bold;">Nota</p>
<p>Em resumo, modelos lineares encontram um conjunto de pesos para combinar
features linearmente e prever o alvo. Por exemplo, o modelo pode chegar a uma
regra como:</p>
<ul class="simple">
<li>se <tt class="docutils literal">0.1 * age + 3.3 * <span class="pre">hours-per-week</span> - 15.1 &gt; 0</tt>, prevê <tt class="docutils literal"><span class="pre">high-income</span></tt></li>
<li>caso contrário, prevê <tt class="docutils literal"><span class="pre">low-income</span></tt></li>
</ul>
<p class="last">Modelos lineares, e em particular a regressão logística, serão
abordados em mais detalhes no módulo "Modelos lineares" mais adiante neste
curso. Por enquanto, o foco é usar esse modelo de regressão logística no
scikit-learn, em vez de entender em detalhes como ele funciona.</p>
</div>

Para criar um modelo de regressão logística no scikit-learn, você pode fazer:

In [14]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

Agora que o modelo foi criado, você pode usá-lo exatamente da mesma forma que
usamos o modelo de k-nearest neighbors no notebook anterior. Em particular,
podemos usar o método `fit` para treinar o modelo usando os dados e rótulos de
treino:

In [15]:
model.fit(data_train, target_train)

LogisticRegression()

Também podemos usar o método `score` para verificar o desempenho de
generalização do modelo no conjunto de teste.

In [16]:
accuracy = model.score(data_test, target_test)
print(f"Acurácia da regressão logística: {accuracy:.3f}")

Acurácia da regressão logística: 0.807


## Recapitulando o notebook

No scikit-learn, o método `score` de um modelo de classificação retorna a
acurácia, ou seja, a fração de amostras classificadas corretamente. Neste caso,
aproximadamente 8 em 10 vezes a regressão logística prevê corretamente a renda
de uma pessoa. Agora a pergunta real é: esse desempenho de generalização é
indicativo de um bom modelo preditivo? Descubra resolvendo o próximo exercício!

Neste notebook, aprendemos a:

* identificar dados numéricos em um conjunto de dados heterogêneo;
* selecionar o subconjunto de colunas correspondente aos dados numéricos;
* usar a função `train_test_split` do scikit-learn para separar os dados em um
  conjunto de treino e um de teste;
* treinar e avaliar um modelo de regressão logística.